In [15]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA

from bokeh.plotting import figure, show
from bokeh.layouts import gridplot
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.palettes import Category20

In [3]:
def read_dataframes(main_dir):
    all_dataframes = []

    for root, dirs, files in os.walk(main_dir):
        for file in files:
            if file.endswith(".xlsx"):
                print(file)
                file_path = os.path.join(root, file)
                target_group = os.path.basename(os.path.dirname(file_path))
                df = pd.read_excel(file_path, dtype={'start_date': str, 'end_date': str})
                df['target_group'] = target_group
                df = df[['target_group', 'daypart', 'total_grp', 'ster_grp', 'regio_grp', 'npo1_grp', 'npo2_grp', 'npo3_grp', 'ster_1+', 'ster_2+', 'ster_3+', 'ster_5+', 'ster_10+','regio_1+', 'regio_2+', 'regio_3+', 'regio_5+', 'regio_10+', 'start_date', 'end_date']]
                all_dataframes.append(df)

    return pd.concat(all_dataframes)

In [4]:
# Main directory containing subdirectories for each target group
main_dir = "../../data/processed/bereik/"

# Concatenate dataframes from all Excel files in the main directory
data = read_dataframes(main_dir)

13+_bereik_all_flights.xlsx
20-49_bereik_all_flights.xlsx
25+_bereik_all_flights.xlsx
25-39_bereik_all_flights.xlsx
25-54_bereik_all_flights.xlsx
25-59_bereik_all_flights.xlsx
25-59AB1_bereik_all_flights.xlsx
25-67_bereik_all_flights.xlsx
25-67AB1_bereik_all_flights.xlsx
3-12_bereik_all_flights.xlsx
30-65_bereik_all_flights.xlsx
35+_bereik_all_flights.xlsx
35-59_bereik_all_flights.xlsx
35-64_bereik_all_flights.xlsx
50+_bereik_all_flights.xlsx
6+_bereik_all_flights.xlsx
68+_bereik_all_flights.xlsx
7-12_bereik_all_flights.xlsx
BDS 25-59_bereik_all_flights.xlsx
BDS 25-67_bereik_all_flights.xlsx
BDS KZ12_bereik_all_flights.xlsx
BDS+KND_bereik_all_flights.xlsx
DAG 25-59_bereik_all_flights.xlsx
M 20-49_bereik_all_flights.xlsx
M 25-54_bereik_all_flights.xlsx
M 25-59_bereik_all_flights.xlsx
M 25-67_bereik_all_flights.xlsx
M 50+_bereik_all_flights.xlsx
V 20-49_bereik_all_flights.xlsx
V 25-54_bereik_all_flights.xlsx
V 25-59_bereik_all_flights.xlsx
V 25-67_bereik_all_flights.xlsx
V 40+_bereik_all

In [10]:
df = data.copy()

df['start_date'] = pd.to_datetime(df['start_date'], format='%d%m%y')
df['end_date'] = pd.to_datetime(df['end_date'], format='%d%m%y')
# Calculate duration in days
df['duration'] = (df['end_date'] - df['start_date']).dt.days + 1


In [11]:
df.head(2)

,target_group,daypart,total_grp,ster_grp,regio_grp,npo1_grp,npo2_grp,npo3_grp,ster_1+,ster_2+,...,ster_5+,ster_10+,regio_1+,regio_2+,regio_3+,regio_5+,regio_10+,start_date,end_date,duration
0,13+,total,63.57,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-01-15,2024-01-28,14
1,13+,primetime,11.30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-01-15,2024-01-28,14


In [66]:
def dbscan_and_pca(df, features, min_samples = 2, eps = 0.3):
    # Aggregating data for each target group
    target_group_data = df.groupby('target_group')[features].mean()

    # Scaling the features
    scaler = StandardScaler()
    target_group_data_scaled = scaler.fit_transform(target_group_data)

    # Clustering using DBSCAN
    dbscan = DBSCAN(eps=eps, min_samples=min_samples, leaf_size=60)  # Example parameters, adjust as needed
    clusters = dbscan.fit_predict(target_group_data_scaled)
    
    # Add cluster labels to DataFrame
    target_group_data['cluster'] = clusters

    # Analyze the clusters
    cluster_counts = target_group_data['cluster'].value_counts()
    print("Cluster Counts:")
    print(cluster_counts)

    # Dimensionality reduction using PCA
    pca = PCA(n_components=2)  # 2 principal components for visualization
    target_group_data_pca = pca.fit_transform(target_group_data_scaled)

    return target_group_data_pca, clusters, target_group_data.index

In [67]:
# Create a color palette for positive cluster labels
positive_color_palette = Category20[20]

# Map negative cluster labels to a specific color
negative_color = "gray"  # You can choose any color for negative cluster labels

def map_cluster_label_to_color(cluster_label):
    if cluster_label >= 0:
        return positive_color_palette[cluster_label % len(positive_color_palette)]
    else:
        return negative_color

def plot_pca_bokeh(df, feature_sets, min_samples, eps):
    plots = []

    # Iterate over feature sets and plot PCA for each
    for i, features in enumerate(feature_sets):
        # Perform DBSCAN and PCA for the current feature set
        target_group_data_pca, clusters, target_groups = dbscan_and_pca(df, features, min_samples, eps)

        # Check if target_group_data_pca contains any elements
        if len(target_group_data_pca) > 0:
            # Create a ColumnDataSource
            source = ColumnDataSource(data=dict(
                x=target_group_data_pca[:, 0],
                y=target_group_data_pca[:, 1],
                target_group=target_groups,
                cluster=clusters
            ))
            
            # Create a ColumnDataSource including x, y coordinates and cluster colors
            data = dict(
                x=source.data['x'],
                y=source.data['y'],
                cluster=source.data['cluster']
            )

            # Create a list of colors based on cluster labels
            cluster_colors = [map_cluster_label_to_color(label) for label in source.data['cluster']]
            data['fill_color'] = cluster_colors

            # Create a ColumnDataSource
            cds = ColumnDataSource(data)

            # Create Bokeh plot
            plot = figure(title=f'(PCA) - Features: {", ".join(features)}',
                          x_axis_label='Principal Component 1', y_axis_label='Principal Component 2',
                          tools='pan,box_zoom,reset,hover', width=800, height=800)
            
            cluster_colors = [map_cluster_label_to_color(label) for label in source.data['cluster']]
            plot.scatter('x', 'y', source=cds, fill_color='fill_color', size=8, fill_alpha=1)
            plot.text('x', 'y', text='target_group', source=source, text_font_size='8pt', text_align='center', text_baseline='bottom', y_offset=16)
            plot.text('x', 'y', text='cluster', source=source, text_font_size='10pt', text_align='center', text_baseline='bottom', y_offset=-10)

            # Add hover tool
            hover = HoverTool()
            hover.tooltips = [('Target Group', '@target_group'), ('Cluster', '@cluster')]
            plot.add_tools(hover)

            plots.append(plot)
            
    # Arrange plots in a grid layout
    num_cols = 2
    num_plots = len(plots)
    num_rows = (num_plots + num_cols - 1) // num_cols
    plots_flattened = [plots[i:i + num_cols] for i in range(0, num_plots, num_cols)]
    grid = gridplot(plots_flattened, width=800, height=800)

    # Show the plot
    show(grid)

In [70]:
feature_sets = [
    ['ster_1+', 'ster_2+'],  
    # ['ster_grp', 'regio_grp'],
    ['ster_1+', 'ster_2+', 'ster_3+', 'ster_10+'],
    ['ster_1+', 'regio_1+'], 
    # ['ster_1+', 'ster_2+', 'ster_grp', 'regio_grp'],
    ['ster_1+', 'ster_2+', 'ster_10+', 'regio_1+', 'regio_2+', 'regio_10+'],
    
]
min_samples = 2
eps = 0.3
plot_pca_bokeh(df, feature_sets, min_samples, eps)


Cluster Counts:
cluster
 0    20
 1     5
 3     3
 4     3
-1     2
 2     2
Name: count, dtype: int64
Cluster Counts:
cluster
 2    18
 1     5
-1     3
 4     3
 0     2
 3     2
 5     2
Name: count, dtype: int64
Cluster Counts:
cluster
 0    20
 1     5
 3     3
 4     3
-1     2
 2     2
Name: count, dtype: int64
Cluster Counts:
cluster
-1    13
 4     7
 3     5
 0     3
 2     3
 1     2
 5     2
Name: count, dtype: int64


In [55]:
# Aggregating data for each target group
# target_group_data = df.groupby('target_group')[['ster_1+', 'ster_grp']].mean()
target_group_data = df.groupby('target_group')[['ster_1+', 'ster_2+', 'ster_3+']].mean()


# Scaling the features
scaler = StandardScaler()
target_group_data_scaled = scaler.fit_transform(target_group_data)

# Clustering using DBSCAN
dbscan = DBSCAN(eps=0.2, min_samples=3)  # Example parameters, adjust as needed
clusters = dbscan.fit_predict(target_group_data_scaled)

# Add cluster labels to DataFrame
target_group_data['cluster'] = clusters

# Analyze the clusters
cluster_counts = target_group_data['cluster'].value_counts()
print("Cluster Counts:")
print(cluster_counts)

# Output clusters and their target groups
for cluster_id, count in cluster_counts.items():
    target_groups_in_cluster = target_group_data[target_group_data['cluster'] == cluster_id].index.tolist()
    print(f"Cluster {cluster_id}: {target_groups_in_cluster}")


# Dimensionality reduction using PCA
pca = PCA(n_components=5)  # 2 principal components for visualization
target_group_data_pca = pca.fit_transform(target_group_data_scaled)

# Visualize clusters
plt.figure(figsize=(14, 10))
scatter = plt.scatter(target_group_data_pca[:, 0], target_group_data_pca[:, 1], c=clusters, cmap='viridis')

# Annotate each point with target group
for i, txt in enumerate(target_group_data.index):
    plt.annotate(txt, (target_group_data_pca[i, 0], target_group_data_pca[i, 1]))

plt.title('DBSCAN Clustering Visualization (PCA)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.colorbar(scatter, label='Cluster')
plt.show()

Cluster Counts:
cluster
 2    15
-1     8
 0     5
 1     4
 3     3
Name: count, dtype: int64
Cluster 2: ['25-59', '25-59AB1', '25-67', '25-67AB1', '30-65', '35-59', '35-64', '6+', 'BDS 25-59', 'BDS 25-67', 'DAG 25-59', 'M 25-59', 'M 25-67', 'V 25-59', 'V 25-67']
Cluster -1: ['13+', '25+', '3-12', '35+', '68+', '7-12', 'V 40+', 'W1 45-75']
Cluster 0: ['20-49', '25-39', 'BDS KZ12', 'M 20-49', 'V 20-49']
Cluster 1: ['25-54', 'BDS+KND', 'M 25-54', 'V 25-54']
Cluster 3: ['50+', 'M 50+', 'V 50+']


ValueError: n_components=5 must be between 0 and min(n_samples, n_features)=3 with svd_solver='full'